# LSTM 预测 Notebook

这个 notebook 用于训练价格预测用的 LSTM，并展示 one-step rolling 评估、更贴近 runtime 的 block forecast 评估，以及 artifact 保存闭环。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

project_root = Path.cwd().resolve()
if not (project_root / "configs").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root


In [ ]:
from forecast.artifacts import get_default_lstm_artifact_paths
from forecast.lstm_forecaster import LSTMForecaster, save_lstm_forecaster_artifacts
from forecast.lstm_model import LSTMPricePredictor
from forecast.notebook_utils import (
    evaluate_block_forecast,
    evaluate_one_step_forecast,
    load_price_series,
    prepare_lstm_data_bundle,
    train_lstm_model,
)


In [ ]:
data_csv = project_root / "data" / "train_prices.csv"
seq_len = 96 * 3
pred_len = 4
train_ratio = 0.7
val_ratio = 0.15
batch_size = 128
epochs = 20
lr = 1e-3
hidden_size = 128
num_layers = 2
dropout = 0.2
runtime_horizon = 24
block_stride = 24
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
price_series = load_price_series(data_csv)
data_bundle = prepare_lstm_data_bundle(
    price_series,
    seq_len=seq_len,
    pred_len=pred_len,
    train_ratio=train_ratio,
    val_ratio=val_ratio,
    batch_size=batch_size,
)
splits = data_bundle["splits"]
scaler = data_bundle["scaler"]
train_loader = data_bundle["train_loader"]
val_loader = data_bundle["val_loader"]
artifact_paths = get_default_lstm_artifact_paths(project_root)
print({key: str(value) for key, value in artifact_paths.items()})
print({name: len(series) for name, series in splits.items()})


In [ ]:
model = LSTMPricePredictor(
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    pred_len=pred_len,
)
train_bundle = train_lstm_model(
    model,
    train_loader,
    val_loader,
    epochs=epochs,
    lr=lr,
    device=device,
)
model = train_bundle["model"]
history = train_bundle["history"]

plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.title("LSTM 训练损失")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.grid(True, linestyle=":")
plt.legend()
plt.show()


In [ ]:
one_step_eval = evaluate_one_step_forecast(
    model,
    history_seed=data_bundle["test_history_seed"],
    target_series=splits["test"],
    scaler=scaler,
    seq_len=seq_len,
    pred_len=pred_len,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    device=device,
)

runtime_like_eval = evaluate_block_forecast(
    model,
    history_seed=data_bundle["test_history_seed"],
    target_series=splits["test"],
    scaler=scaler,
    seq_len=seq_len,
    pred_len=pred_len,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    device=device,
    forecast_horizon=runtime_horizon,
    block_stride=block_stride,
)

print("one_step_metrics =", one_step_eval["metrics"])
print("runtime_like_block_metrics =", runtime_like_eval["metrics"])


In [ ]:
first_block = runtime_like_eval["blocks"][0] if runtime_like_eval["blocks"] else None

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(splits["test"], label="真实价格")
axes[0].plot(one_step_eval["predictions"], label="one-step rolling 预测")
axes[0].set_title("测试集 one-step rolling 预测")
axes[0].set_xlabel("test step")
axes[0].set_ylabel("price")
axes[0].grid(True, linestyle=":")
axes[0].legend()

if first_block is not None:
    axes[1].plot(first_block["target"], label="真实窗口")
    axes[1].plot(first_block["prediction"], label="runtime-like block 预测")
    axes[1].set_title(f"首个 block 窗口预测（horizon={runtime_horizon}）")
    axes[1].set_xlabel("window step")
    axes[1].set_ylabel("price")
    axes[1].grid(True, linestyle=":")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "没有可展示的 block forecast", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
artifact_result = save_lstm_forecaster_artifacts(
    model_path=artifact_paths["model_path"],
    state_dict=model.state_dict(),
    scaler=scaler,
    seq_len=seq_len,
    pred_len=pred_len,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
)
runtime_forecaster = LSTMForecaster.from_artifacts(artifact_result["model_path"], device=device)
runtime_window = runtime_forecaster.predict(data_bundle["test_history_seed"], horizon=runtime_horizon + 1)
print({key: str(value) for key, value in artifact_result.items()})
print("runtime forecaster seq_len =", runtime_forecaster.seq_len)
print("runtime forecast window shape =", tuple(runtime_window.shape))
